# DSPy Agents — ReAct with Real Tools

**Week 6 | Notebook 4 of 6**

**What you'll learn:**
- Defining tool functions with docstrings
- Building a `dspy.ReAct` agent
- Running the agent on multi-step questions
- Inspecting the Thought/Action/Observation trace
- MCP tool integration (2025 feature)
- Optimizing the agent with BootstrapFewShot
- Adding Assert constraints to validate tool usage

**Runtime:** ~45 minutes

In [ ]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/04_react_agent.ipynb")

## 1. Setup

In [ ]:
import dspy

from src.config import get_dspy_lm

lm = get_dspy_lm()
dspy.configure(lm=lm)

## 2. Defining Tool Functions

In [ ]:
def search_web(query: str) -> str:
    """Search the web for current information."""
    # Simulated search
    return f"Search results for '{query}': India GDP 2024 = $3.9T, population = 1.4B"


def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error: invalid expression"


def get_weather(city: str) -> str:
    """Get current weather for a city."""
    return f"Weather in {city}: 25°C, sunny"


print("✅ Tools defined:")
print(f"  - search_web: {search_web.__doc__}")
print(f"  - calculator: {calculator.__doc__}")
print(f"  - get_weather: {get_weather.__doc__}")

## 3. Building a ReAct Agent

In [ ]:
# ReAct: Thought → Action → Observation loop
agent = dspy.ReAct("question -> answer", tools=[search_web, calculator, get_weather], max_iters=5)

print("✅ ReAct agent built with 3 tools")
print("Max iterations: 5 (prevents infinite loops)")

## 4. Running Multi-Step Questions

In [ ]:
# Question requiring search + calculation
question = "What is the GDP per capita of India in 2024?"

result = agent(question=question)

print(f"Question: {question}")
print(f"Answer: {result.answer}")

## 5. Inspecting the Thought/Action/Observation Trace

In [ ]:
# DSPy ReAct exposes the trace internally
# For detailed inspection, use dspy.inspect_history()

lm.inspect_history(n=10)  # Show last 10 LLM calls

print("\n🔍 The trace shows:")
print("  1. Thought: reasoning about what to do next")
print("  2. Action: which tool to call with what args")
print("  3. Observation: result from the tool")
print("  4. Repeat until answer or max_iters reached")

## 6. MCP Tool Integration (2025)

In [ ]:
# MCP (Model Context Protocol) tools are supported in DSPy 2025+
# These allow standardized tool definitions across frameworks

mcp_tool_example = {
    "name": "calculator",
    "description": "Evaluate mathematical expressions",
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "Math expression"}},
        "required": ["expression"],
    },
}

print("MCP Tool Definition:")
print(mcp_tool_example)

print("\n💡 MCP standardizes tool definitions across:")
print("   DSPy, LangChain, OpenAI, Anthropic, and more")

## 7. Optimizing with BootstrapFewShot

In [ ]:
from dspy.teleprompt import BootstrapFewShot

# Create training examples for the agent
train_examples = [
    dspy.Example(question="What is 15 * 23?", answer="345").with_inputs("question"),
    dspy.Example(question="What is 100 divided by 4?", answer="25").with_inputs("question"),
]


def agent_metric(example, prediction, trace=None):
    return 1.0 if example.answer in prediction.answer else 0.0


teleprompter = BootstrapFewShot(metric=agent_metric, max_bootstrapped_demos=2)
optimized_agent = teleprompter.compile(agent, trainset=train_examples)

print("✅ Agent optimized with BootstrapFewShot")
print("The optimizer generated few-shot demos for better tool use.")

## 8. Adding Assert Constraints

In [ ]:
class ValidatedAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.agent = dspy.ReAct(
            "question -> answer", tools=[search_web, calculator, get_weather], max_iters=5
        )

    def forward(self, question):
        result = self.agent(question=question)

        # Validate: answer should not be empty
        dspy.Assert(len(result.answer.strip()) > 0, "Agent must provide a non-empty answer.")

        # Validate: answer should be concise
        dspy.Suggest(len(result.answer.split()) <= 100, "Answer should be under 100 words.")

        return result


validated = ValidatedAgent()
try:
    result = validated(question="What is 15 * 23?")
    print(f"Answer: {result.answer}")
except AssertionError as e:
    print(f"Validation failed: {e}")

## 9. Exercise: Build a Financial Analysis Agent

Create a ReAct agent with 5 tools for financial analysis:
1. `get_stock_price(ticker)`
2. `calculate_pe_ratio(price, earnings)`
3. `get_company_info(ticker)`
4. `compare_stocks(ticker1, ticker2)`
5. `get_news(ticker)`

In [ ]:
# YOUR TURN: Financial analysis agent

# def get_stock_price(ticker: str) -> str:
#     """Get current stock price."""
#     return f"${random.uniform(50, 500):.2f}"

# financial_agent = dspy.ReAct(
#     "question -> analysis",
#     tools=[get_stock_price, ...],
#     max_iters=5
# )

# result = financial_agent(question="Analyze Apple stock")
# print(result.analysis)

---

**Next:** [05_gepa_optimizer.ipynb](05_gepa_optimizer.ipynb) — Genetic-Pareto prompt evolution